In [1]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 667, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 667 (delta 174), reused 100 (delta 97), pack-reused 434 (from 3)
Receiving objects: 100% (667/667), 220.60 KiB | 11.03 MiB/s, done.
Resolving deltas: 100% (348/348), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 180 packages in 1.32s
Prepared 10 packages in 1.35s
Uninstalled 4 packages in 200ms
Installed 10 packages in 68ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.1
 - holoviews==1.22.1
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - panel==1.9.3
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do 

In [ ]:
from sklearn.neural_network import MLPClassifier
from cuml.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import pandas as pd
import xgboost as xgb
import cudf
import numpy as np
import json
from pathlib import Path



In [4]:
#Constantes
ARCHIVO_ORIGEN = "Primeras muestras.csv"
ARCHIVO_SALIDA = "dataset_completo.csv"
COLUMNA_CLASE  = "Tipo"

# Los 18 canales espectrales
CANALES = ["A_410", "B_435", "C_460", "D_485", "E_510", "F_535",
           "G_560", "H_585", "R_610", "I_645", "S_680", "J_705",
           "T_730", "U_760", "V_810", "W_860", "K_900", "L_940"]

N_SINTETICAS   = 953        #Esto nos genera unas 40036 muestras
SIGMA          = 0.01
RUIDO_RELATIVO = False      # Lo ponemos a false para que no mantengga la estructura estadística
SEMILLA        = 42

rng = np.random.default_rng(SEMILLA)

df = pd.read_csv(Path(ARCHIVO_ORIGEN), encoding="latin-1", sep=";", decimal=",")

X = df[CANALES].to_numpy(dtype=np.float64)
y = df[COLUMNA_CLASE].to_numpy()


filas_sinteticas, etiquetas_sinteticas = [], []
for firma, etiqueta in zip(X, y):
    sigma_vec = SIGMA * np.abs(firma) if RUIDO_RELATIVO else SIGMA
    ruido = rng.normal(loc=0.0, scale=sigma_vec, size=(N_SINTETICAS, X.shape[1]))
    filas_sinteticas.append(firma + ruido)
    etiquetas_sinteticas.extend([etiqueta] * N_SINTETICAS)

X_sint = np.vstack(filas_sinteticas)

X_final = np.vstack([X, X_sint])
y_final = np.concatenate([y, etiquetas_sinteticas])

df_final = pd.DataFrame(X_final, columns=CANALES)
df_final[COLUMNA_CLASE] = y_final
df_final = df_final.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

df_final.to_csv(Path(ARCHIVO_SALIDA), index=False, encoding="utf-8-sig")

print(f"Firmas reales:       {len(df)}")
print(f"Muestras sintéticas: {len(X_sint)}")
print(f"Total dataset:       {len(df_final)}")
print(df_final[COLUMNA_CLASE].value_counts())

Firmas reales:       36
Muestras sintéticas: 34308
Total dataset:       34344
Tipo
Miel de milflores    13356
Miel Jaramago         5724
Miel de Bosque        5724
Miel de Retama        5724
Miel Sintética        3816
Name: count, dtype: int64


In [8]:
df = pd.read_csv("dataset_completo.csv")

COLS_ELIMINAR = ['Tipo']
X = df.drop(COLS_ELIMINAR, axis=1)
y = df['Tipo']

class_counts = y.value_counts()
clases_validas = class_counts[class_counts >= 10].index
mask = y.isin(clases_validas)

X_filt = X[mask].copy()
y_filt = y[mask].copy()

clases_eliminadas = class_counts[class_counts < 10].index.tolist()

nan_cols = X_filt.columns[X_filt.isna().any()].tolist()


le = LabelEncoder()
y_enc = le.fit_transform(y_filt).astype(np.int32)


X_train, X_test, y_train, y_test = train_test_split(
    X_filt, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_imp).astype(np.float32)


In [9]:
mlp = MLPClassifier(
    hidden_layer_sizes=(100,50,50),
    activation='tanh',
    solver='sgd',
    max_iter=1000,
    alpha = 0.001,
    random_state=42,
    learning_rate='adaptive',
    warm_start = True
)

mlp.fit(X_train, y_train)

predicciones = mlp.predict(X_test)

precision = accuracy_score(y_test, predicciones)
print(f"La precisión de la red es de: {precision * 100}%")

La precisión de la red es de: 98.12199737953122%


In [14]:
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss'
)

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv = 5, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train, y_train)

print(f"Mejor configuración: {grid_search.best_params_}")
print(f"Mejor score de validación: {grid_search.best_score_:.4f}")

y_pred = grid_search.best_estimator_.predict(X_test)
print("accuracy:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred, average='weighted'))
print("recall:", recall_score(y_test, y_pred, average='weighted'))
print("f1:", f1_score(y_test, y_pred, average='weighted'))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:34:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Mejor configuración: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
Mejor score de validación: 0.9833
accuracy: 0.9818023001892561
precision: 0.9818167554693221
recall: 0.9818023001892561
f1: 0.9818077268237452


In [16]:
rf_model = RandomForestClassifier(random_state = 42)
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

grid_rf= GridSearchCV(estimator=rf_model,
                      param_grid=param_grid,
                      scoring='accuracy',
                      cv=5,
                      n_jobs=-1,
                      verbose = 2)

grid_rf.fit(X_train, y_train)


print(f"Mejor combinación: {grid_rf.best_params_}")
print(f"Mejor precisión (CV): {grid_rf.best_score_:.4f}")

y_pred = grid_rf.best_estimator_.predict(X_test)
print("accuracy:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred, average='weighted'))
print("recall:", recall_score(y_test, y_pred, average='weighted'))
print("f1:", f1_score(y_test, y_pred, average='weighted'))

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
540 fits failed out of a total of 2160.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
540 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/cuml/internals/o

Mejor combinación: {'bootstrap': False, 'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
Mejor precisión (CV): 0.9813
accuracy: 0.9813655553937982
precision: 0.9813690477339052
recall: 0.9813655553937982
f1: 0.9813530865112222


In [ ]:
model_XGB = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        tree_method='hist',
        device='cuda',
        random_state=42
    )

model_XGB.fit(X_train_scaled, y_train)
model_XGB.save_model('best_model.json')

clases_lista = model_XGB.classes_.tolist()

with open('clases.json', 'w', encoding='utf-8') as archivo_json:
    json.dump(clases_lista, archivo_json, ensure_ascii=False, indent=4)